In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import zipfile
import io
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

In [2]:
#to load the data
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
resp = urllib.request.urlopen(url)
z = zipfile.ZipFile(io.BytesIO(resp.read()))
raw = z.read("SMSSpamCollection").decode("utf-8")


In [3]:
print (raw[:1000])

ham	Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham	Ok lar... Joking wif u oni...
spam	Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
ham	U dun say so early hor... U c already then say...
ham	Nah I don't think he goes to usf, he lives around here though
spam	FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
ham	Even my brother is not like to speak with me. They treat me like aids patent.
ham	As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
spam	WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
spam	H

In [4]:
lines = raw.strip().split("\n")
labels, texts = [], []
for line in lines:
    label, text = line.split("\t", 1)
    labels.append(1 if label == "spam" else 0)
    texts.append(text)

print(f"{len(texts)} messages, {sum(labels)} spam")

5574 messages, 747 spam


In [5]:
#simply word based tokenisation
def tokenize(text):
    return text.lower().split()

#set of all the words:

all_words = set()
for t in texts:
    all_words.update(tokenize(t))
words=sorted(list(all_words))
print(words[:100])

['!', '!!', '!!!', '!!!!', "!!''.", '!1', '!:-)', '!this', '"', '".', '"1.u', '"3000', '"a', '"adp"', '"ah', '"alright', '"alrite', '"are', '"aww', '"be', '"because', '"best', '"boo', '"boost', '"can', '"cha', '"checkmate"', '"cheers', '"crazy"', '"divorce', '"don\'t', '"drink".', '"drive', '"enjoy"', '"er,', '"ey!', '"find', '"for', '"get', '"getting', '"gimme', '"go', '"goodmorning', '"gran', '"gud', '"happy', '"hello', '"hello"', '"hello-/@drivby-:0quit', '"hey', '"hey!', '"hi', '"how', '"hurt', '"i', '"i;m', '"im', '"it', '"its', '"jeevithathile', '"julianaland"', '"keep', '"kudi"yarasu', '"life', '"margaret', '"me', '"miss', '"mix"', '"morning"', '"none!nowhere', '"not', '"nver', '"oh', '"oh"', '"our', '"paths', '"pete', '"petey', '"polys"', '"power', '"response"', '"shah', '"she', '"shit', '"si.como', '"sleep', '"smokes', '"sometimes', '"song', '"speak', '"stop', '"suppliers",', '"sweet"', '"symptoms"', '"the', '"thinking', '"this', '"ur', '"urgent!', '"usf']


In [6]:
vocab = sorted(words)
stoi = {w: i+1 for i, w in enumerate(vocab)}  # +1 reserves 0 for padding
stoi["<pad>"] = 0
itos = {i: w for w, i in stoi.items()}
vocab_size = len(stoi)

print(f"vocab size: {vocab_size}")

vocab size: 13628


In [7]:
def encode(text):
    return [stoi[w] for w in tokenize(text)]

def decode(ids):
    return " ".join(itos[i] for i in ids)
print(encode(texts[0]))
print(decode(encode(texts[0])))

[5469, 12430, 6776, 9313, 3636, 2108, 8746, 6360, 2749, 8202, 5603, 13163, 6983, 4369, 2747, 3238, 11790, 5561, 1783, 12781]
go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat...


In [8]:
#to encode all the msgs
encoded = [torch.tensor(encode(t)) for t in texts]
lengths_all = torch.tensor([len(e) for e in encoded])
y = torch.tensor(labels, dtype=torch.long)
#padding all sequences to the length of the longest
X = pad_sequence(encoded, batch_first=True, padding_value=stoi["<pad>"])
print(X.shape)



torch.Size([5574, 171])


In [9]:
#80% training split
n = int(0.8 * len(X))
perm = torch.randperm(len(X))
X, y, lengths_all = X[perm], y[perm], lengths_all[perm]  # shuffled together, once
Xtr, ytr, lengths_tr = X[:n], y[:n], lengths_all[:n]
Xval, yval, lengths_val = X[n:], y[n:], lengths_all[n:]

In [10]:
#now we will implement an lstm model
#following the emebedding pattern as shown in karpathy's GPT model

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0) #vocab_sie x embed dim size table
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
      emb = self.embed(x)                  # (B, T, embed_dim)
      out, (h_n, c_n) = self.lstm(emb)      # h_n: (1, B, hidden_size)
      final_hidden = h_n.squeeze(0)         # (B, hidden_size)
      return self.head(final_hidden).squeeze(-1)
#h_n, c_n are the final hidden and cell states respectively
#B is the batch size, T is the count of tokens.

In [11]:
model = LSTMClassifier(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
'''
for step in range(200):
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step}: loss {loss.item():.4f}")
        '''

'\nfor step in range(200):\n    logits = model(Xtr)\n    loss = loss_fn(logits, ytr)\n    optimizer.zero_grad()\n    loss.backward()\n    optimizer.step()\n    if step % 50 == 0:\n        print(f"step {step}: loss {loss.item():.4f}")\n        '

In [12]:
#slightly inefficient for the lstm model since it uneccesarily keeps processing 0s due to padding.
#hence we will pack all the data to return padded data back to original data
#why did we have to pad in the first place? bcus unlike when we were dealing with just chars as tokens, works all have different sizes and to iterate in batches they all need to have the same size.

In [13]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        emb = self.embed(x)  # (B, T, embed_dim)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, c_n) = self.lstm(packed)
        final_hidden = h_n.squeeze(0)     # (B, hidden_size)
        logits = self.head(final_hidden)  # (B, num_classes)
        return logits

model = LSTMClassifier(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)


In [14]:
#working on batches instead of whole dataset together
def get_batch(X, y, lengths, batch_size=32):
    ix = torch.randint(len(X), (batch_size,))
    return X[ix], y[ix], lengths[ix]

In [15]:
@torch.no_grad()
def evaluate(model, X, y, lengths):
    model.eval()
    logits = model(X, lengths)
    loss = F.cross_entropy(logits, y)
    preds = logits.argmax(dim=-1)
    acc = (preds == y).float().mean().item()
    model.train()
    return loss.item(), acc

In [16]:
batch_size = 32
max_steps = 1000
eval_interval = 200

for step in range(max_steps):
    xb, yb, lengths_b = get_batch(Xtr, ytr, lengths_tr, batch_size)

    logits = model(xb, lengths_b)
    loss = F.cross_entropy(logits, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        val_loss, val_acc = evaluate(model, Xval, yval, lengths_val)
        print(f"step {step:4d}: train_loss {loss.item():.4f}, val_loss {val_loss:.4f}, val_acc {val_acc:.4f}")

# final full evaluation
val_loss, val_acc = evaluate(model, Xval, yval, lengths_val)
print(f"\nfinal: val_loss {val_loss:.4f}, val_acc {val_acc:.4f}")

step    0: train_loss 0.7066, val_loss 0.6922, val_acc 0.5130
step  200: train_loss 0.1522, val_loss 0.2167, val_acc 0.9247
step  400: train_loss 0.0362, val_loss 0.1204, val_acc 0.9614
step  600: train_loss 0.0099, val_loss 0.1159, val_acc 0.9641
step  800: train_loss 0.0066, val_loss 0.0999, val_acc 0.9695

final: val_loss 0.1017, val_acc 0.9767


In [17]:
#predicting on new text examples
def predict(text):
    model.eval()
    words = tokenize(text)
    ids = [stoi.get(w, stoi["<pad>"]) for w in words]  # unseen words -> <pad>/0 (crude fallback)
    ids = torch.tensor(ids).unsqueeze(0)               # (1, T)
    length = torch.tensor([ids.shape[1]])
    with torch.no_grad():
        logits = model(ids, length)              # (1, 2)
        probs = F.softmax(logits, dim=-1)
        pred_class = probs.argmax(dim=-1).item()
    model.train()
    return ("SPAM" if pred_class == 1 else "HAM"), probs.squeeze().tolist()

print(predict("congratulations you have won a free iphone click now"))
print(predict("hey are we still on for lunch tomorrow"))

('SPAM', [0.13879933953285217, 0.8612006902694702])
('HAM', [0.9995666146278381, 0.00043342652497813106])


In [18]:
print(predict("hi, how are you"))

('HAM', [0.9989557266235352, 0.0010442595230415463])


In [20]:
 predict("do you want to get coffee?")

('HAM', [0.9995501637458801, 0.00044984472333453596])

In [24]:
print(predict("Rs 500 Off on Amazon Prime! Join Amazon Prime at Rs999/year instead of Rs 1499. Enjoy blockbuster entertainment, shopping deals & free 1-day delivery with Vi Max Postpaid. *Limited period offer. Discount will reflect in your current or next bill. T&C apply."))

('SPAM', [0.0005049906321801245, 0.9994950294494629])


In [25]:
print(predict("I am in society meeting will call in sometime"))

('HAM', [0.9997673630714417, 0.00023268329096026719])


In [26]:
[print(predict("Dear customer thank you for shopping with Pantaloons. Please click the link to view your e-bill "))]

('SPAM', [0.0058172899298369884, 0.9941827654838562])


[None]

In [ ]:
D